#### Libraries, plot settings, utils

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
from scipy.integrate import odeint

# ---------------------------------------------------------------------
# Plot configuration
# ---------------------------------------------------------------------
plt.style.use("tableau-colorblind10")
cmap = plt.get_cmap("viridis")  # Colormap

plt.rcParams["figure.autolayout"] = True
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9

plt.rcParams.update({
    "font.size": 10,
    "font.family": "serif",
    "font.serif": "cmr10",
    "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True,
})

print(plt.style.available)

# ---------------------------------------------------------------------
# Network dynamics
# ---------------------------------------------------------------------

def phi(x):
    return np.tanh(x)

def dphi(x):
    return 1 - np.tanh(x)**2

def dynamics(x, t, W):
    return -x + np.dot(W, phi(x))

# ---------------------------------------------------------------------
# Network structure
# ---------------------------------------------------------------------

def inter_matrix(K, N, g, gamma):
    J = np.zeros([N, N])
    for i in range(N):
        for j in range(i + 1, N):
            z = np.random.normal(size=2)
            J[i, j] = g / np.sqrt(K) * z[0]
            J[j, i] = g / np.sqrt(K) * (gamma * z[0] + np.sqrt(1 - gamma**2) * z[1])
    return J

def adjacency_matrix(N, K, degree_sequence):
    A = np.zeros([N, N])
    for i in range(N):
        for j in range(i + 1, N):
            p_ij = degree_sequence[i] * degree_sequence[j] / (K * N)
            if np.random.rand() < p_ij:
                A[i, j] = 1
                A[j, i] = 1
    return A

def sample_lognormal(mu, sigma, size=1):
    return np.random.lognormal(mean=mu, sigma=sigma, size=size)

# ---------------------------------------------------------------------
# Simulation
# ---------------------------------------------------------------------

def simulate_lognormal(params, hyperparams, ic=None, path_to_save=None):
    N, g, gamma, mu, sigma = params
    N_steps, dt = hyperparams
    t = np.linspace(0, N_steps * dt, N_steps)

    degree_sequence = sample_lognormal(mu, sigma, N)
    K = int(np.mean(degree_sequence))
    params.append(K)

    A = adjacency_matrix(N, K, degree_sequence)
    J = inter_matrix(K, N, g, gamma)
    W = A * J

    if ic is None:
        sim = odeint(dynamics, np.random.random(N) * 2 - 1, t, args=(W,))
    else:
        sim = odeint(dynamics, ic, t, args=(W,))

    return sim, degree_sequence

#### Critical line

In [ ]:
from scipy.stats import lognorm
from scipy.optimize import fsolve

# ---------------------------------------------------------------------
# Functions for theoretical critical g
# ---------------------------------------------------------------------

def chi(K, k_list, Pk_list, gamma, g):
    return g * np.sum(Pk_list * (k_list / K) ** (3 / 2) * 1 / (1 - gamma * g**2 * k_list / K))

def critical_g(g, K, k_list, Pk_list, gamma):
    chi_val = chi(K, k_list, Pk_list, gamma, g)
    return 1 - g**2 * np.sum(Pk_list * (k_list / K) ** 2 / (1 - gamma * g**2 * chi_val * k_list / K) ** 2)

# ---------------------------------------------------------------------
# Degree distribution
# ---------------------------------------------------------------------

k_list = np.arange(1, 1000)
sigma = 1
mean = 3

# Log-normal degree distribution (scipy parameterization)
Pk_list = lognorm.pdf(k_list, s=sigma, scale=np.exp(mean))
K = np.sum(k_list * Pk_list)

# ---------------------------------------------------------------------
# Compute theoretical critical line
# ---------------------------------------------------------------------

gamma_list_theory = np.linspace(0, 1, 200_000)
g_list_theory = []

g_crit = 0.6  # initial guess
for gamma in gamma_list_theory:
    g_crit = fsolve(
        critical_g, g_crit,
        args=(K, k_list, Pk_list, gamma),
        xtol=1e-14, maxfev=1000
    )[0]
    g_list_theory.append(g_crit)

#### Simulations for plots 

In [ ]:
from joblib import Parallel, delayed

# ------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------
params = [
    [4000, 0.1, 0.5, 3, 1],
    [4000, 3,   0.9, 3, 1],
    [4000, 3,   0.5, 3, 1],
    [4000, 3,   0.1, 3, 1]
]

hyperparams = [500, 0.1]  # N_steps, dt
n_params = len(params)
n_iters = 5
n_nodes = params[0][0]

# ------------------------------------------------------------------
# Storage
# ------------------------------------------------------------------
sim_list = [[] for _ in range(n_params)]
degree_sequences = [[] for _ in range(n_params)]
stat = np.zeros((n_params, n_nodes * n_iters))

# ------------------------------------------------------------------
# Parallel worker
# ------------------------------------------------------------------
def run_sim(j, param, hyperparams, ic):
    sim, degree_sequence = simulate_lognormal(param, hyperparams, ic=ic)
    return j, sim, degree_sequence

# ------------------------------------------------------------------
# Main loop
# ------------------------------------------------------------------
for i in range(n_iters):
    print(f"Iteration {i}")

    # Initial conditions
    ics = [None] * n_params if i == 0 else [sim_list[j][-1][-1] for j in range(n_params)]

    # Run parameter sweeps in parallel
    results = Parallel(n_jobs=-1)(
        delayed(run_sim)(j, params[j], hyperparams, ics[j]) for j in range(n_params)
    )

    # Collect results
    for j, sim, degree_sequence in results:
        sim_list[j].append(sim)
        degree_sequences[j].append(degree_sequence)
        stat[j, i*n_nodes:(i+1)*n_nodes] = sim[-1]

# ------------------------------------------------------------------
# Done
# ------------------------------------------------------------------
print("Simulation finished.")
print("stat shape:", stat.shape)

#### Plot phase diagram

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ------------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------------
if 'sim_list' not in globals() or 'degree_sequences' not in globals():
    raise NameError("Run the simulation cell first: sim_list and degree_sequences must exist.")

# ------------------------------------------------------------------
# Colors
# ------------------------------------------------------------------
colors = ['royalblue', 'firebrick', 'forestgreen', 'indigo', 'goldenrod', 'darkorange']
colors_trajectory = ['#D64444', '#43A4C4']

# ------------------------------------------------------------------
# Theoretical curve
# ------------------------------------------------------------------
gamma_list_theory = np.asarray(gamma_list_theory)
g_list_theory = np.asarray(g_list_theory)
order = np.argsort(gamma_list_theory)
gamma_list_theory = gamma_list_theory[order]
g_list_theory = g_list_theory[order]

g_left, g_right = 0, 4

# ------------------------------------------------------------------
# Figure & grid
# ------------------------------------------------------------------
fig = plt.figure(figsize=(18/2.54, 6/2.54))
gs = GridSpec(1, 2, width_ratios=[1.05, 1.95], wspace=0.22)

# ------------------------------------------------------------------
# Left panel — phase diagram
# ------------------------------------------------------------------
ax_left = fig.add_subplot(gs[0])

gamma_vals = np.linspace(gamma_list_theory.min(), gamma_list_theory.max(), 400)
g_of_gamma = np.interp(gamma_vals, gamma_list_theory, g_list_theory)

# Phase regions
ax_left.fill_betweenx(gamma_vals, g_left, g_of_gamma, color=colors[0], alpha=0.25)  # (i)
gamma_cut = 0.6
mask_bottom = gamma_vals <= gamma_cut
mask_top = gamma_vals > gamma_cut
ax_left.fill_betweenx(gamma_vals[mask_bottom], g_of_gamma[mask_bottom], g_right, color=colors[1], alpha=0.25)  # (iv)
ax_left.fill_betweenx(gamma_vals[mask_top], g_of_gamma[mask_top], g_right, color=colors[5], alpha=0.25)  # (ii)/(iii)

# Theoretical boundary
ax_left.plot(g_list_theory, gamma_list_theory, color='black', linewidth=2)

# Horizontal intersection at gamma_cut
if gamma_list_theory.min() <= gamma_cut <= gamma_list_theory.max():
    g_intersect = np.interp(gamma_cut, gamma_list_theory, g_list_theory)
    ax_left.hlines(gamma_cut, g_intersect, g_right, color='black', linestyle='-.', linewidth=1.2)

# Formatting
ax_left.set_xlabel(r'$g$')
ax_left.set_ylabel(r'$\gamma$')
ax_left.set_xlim(g_left, g_right)
ax_left.set_ylim(gamma_list_theory.min(), gamma_list_theory.max())
ax_left.set_box_aspect(1)
ax_left.spines['top'].set_visible(False)
ax_left.spines['right'].set_visible(False)

dx = g_right - g_left
ax_left.text(g_left + -0.01*dx, 0.47, '(i)', fontweight='bold')
ax_left.text(g_left + 0.7*dx, 0.87, '(ii)', fontweight='bold')
ax_left.text(g_left + 0.7*dx, 0.47, '(iii)', fontweight='bold')
ax_left.text(g_left + 0.7*dx, 0.07, '(iv)', fontweight='bold')

# ------------------------------------------------------------------
# Right panel — 2×2 simulation grid
# ------------------------------------------------------------------
gs_right = gs[1].subgridspec(2, 2, wspace=0.28, hspace=0.40)
axes = [fig.add_subplot(gs_right[i]) for i in range(4)]
N_last = 100
labels_roman = ['(i)', '(ii)', '(iii)', '(iv)']

for i, (sim, degrees) in enumerate(zip(sim_list, degree_sequences)):
    ax = axes[i]
    sim = sim[0]
    degrees = degrees[0]
    if sim.shape[0] < sim.shape[1]:
        sim = sim.T
    N_neurons, _ = sim.shape

    y_min, y_max = (-1, 1) if i == 0 else (-10, 10)

    if len(degrees) == N_neurons:
        median = np.median(degrees)
        bin1, bin2 = np.where(degrees <= median)[0], np.where(degrees > median)[0]
        sel1 = np.random.choice(bin1, min(5, len(bin1)), replace=False)
        sel2 = np.random.choice(bin2, min(5, len(bin2)), replace=False)
        for idx in sel1: ax.plot(sim[idx], color=colors_trajectory[0], alpha=0.85)
        for idx in sel2: ax.plot(sim[idx], color=colors_trajectory[1], alpha=0.85)

        stat1 = sim[bin1, -N_last:].mean(axis=1)
        stat2 = sim[bin2, -N_last:].mean(axis=1)
        inset = ax.inset_axes([1., 0.05, 0.25, 0.90])
        inset.hist(stat1, bins=10, orientation='horizontal', color=colors_trajectory[0], alpha=0.7, density=True)
        inset.hist(stat2, bins=[50, 60, 40, 20][i], orientation='horizontal', color=colors_trajectory[1], alpha=0.7, density=True)

    else:
        sel = np.random.choice(N_neurons, min(5, N_neurons), replace=False)
        for idx in sel: ax.plot(sim[idx], color=colors_trajectory[1], alpha=0.85)
        stat = sim[:, -N_last:].mean(axis=1)
        inset = ax.inset_axes([1.08, 0.05, 0.20, 0.90])
        inset.hist(stat, bins=30, orientation='horizontal', color=colors_trajectory[1], alpha=0.6, density=True)

    if i == 0:
        inset.legend(loc='upper center', bbox_to_anchor=(-2, 1.2), frameon=False, ncol=1)

    inset.set_ylim(y_min, y_max)
    inset.set_xticks([]); inset.set_yticks([])
    for spine in inset.spines.values(): spine.set_visible(False)

    ax.text(-0.10, 1.02, labels_roman[i], transform=ax.transAxes, ha='left', va='bottom', fontweight='bold')
    ax.set_ylim(y_min, y_max)
    ax.set_xticks([]); ax.set_yticks([])
    if i // 2 == 1: ax.text(0.5, -0.08, r'$t$', transform=ax.transAxes, ha='center', va='top')
    if i % 2 == 0: ax.text(-0.12, 0.5, r'$x(t)$', transform=ax.transAxes, ha='right', va='center', rotation=90)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.savefig('PLACEHOLDER/phase_diagram_gamma_top.png', dpi=1000, bbox_inches='tight')

#### Loading numerical validation data

In [ ]:
# ---------------------------------------------------------------------
# Replica analysis
# ---------------------------------------------------------------------

# Load replica data
with open("repliche_N4000.pkl", "rb") as f:
    results_loaded = pickle.load(f)

df = pd.DataFrame(results_loaded, columns=["N", "g", "gamma", "mu", "sigma", "n", "d"])

# Temporal statistics after burn-in
def temporal_mean(arr_list, burn_in=20_000):
    """Compute mean across arrays after burn-in."""
    arr = np.hstack([a[burn_in:] for a in arr_list])
    return arr.mean()

def temporal_std(arr_list, burn_in=20_000):
    """Compute std across arrays after burn-in."""
    arr = np.hstack([a[burn_in:] for a in arr_list])
    return arr.std()

# Aggregation by parameters
burn_in = 20_000
grouped = (
    df.groupby(["N", "gamma"])
      .agg(
          avg_d=("d", lambda x: temporal_mean(x, burn_in=burn_in)),
          std_d=("d", lambda x: temporal_std(x, burn_in=burn_in))
      )
      .reset_index()
)

# ---------------------------------------------------------------------
# Largest Lyapunov exponent (LLE) analysis
# ---------------------------------------------------------------------

df_LLE = pd.read_pickle("PLACEHOLDER/df_LLe.pkl")

pivot = df_LLE.pivot(
    index="gamma",   # y-axis
    columns="g",     # x-axis
    values="Z"
)

g_vals = pivot.columns.values
gamma_vals = pivot.index.values
Zmat = pivot.values

# ---------------------------------------------------------------------
# Note: Largest real part of eigenvalues analysis (g_zero_df) 
# TO BE ADDED
# ---------------------------------------------------------------------
# g_zero_df = pd.read_pickle('PLACEHOLDER/g_zero_df.pkl')

#### Numerical validations of critical lines

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# Colormap and color limits
# ---------------------------------------------------------------------
cmap = "RdBu_r"
vmin = df_LLE["Z"].min()
vmax = df_LLE["Z"].max()

fig, axes = plt.subplots(
    1, 2,
    figsize=(18 / 2.54, 6 / 2.54),
    gridspec_kw={"width_ratios": [1, 2]}
)

# ---------------------------------------------------------------------
# LEFT PLOT: g vs gamma heatmap
# ---------------------------------------------------------------------
ax = axes[0]

pivot_left = df_LLE.pivot(index="gamma", columns="g", values="Z")
g_vals = pivot_left.columns.values
gamma_vals = pivot_left.index.values
Zmat = pivot_left.values

im_left = ax.imshow(
    Zmat,
    origin="lower",
    aspect="auto",
    extent=[g_vals[0], g_vals[-1], gamma_vals[0], gamma_vals[-1]],
    cmap=cmap,
    alpha=1,
)

ax.errorbar(
    g_zero_df["g_zero"],
    g_zero_df["gamma"],
    xerr=g_zero_df["g_zero_error"],
    fmt="o",
    markersize=3,
    capsize=3,
    color="firebrick",
    label="Numerical",
)

ax.plot(
    g_list_theory,
    gamma_list_theory,
    color="black",
    lw=2,
    label="Theoretical curve",
)

ax.set_xlabel(r"$g$")
ax.set_ylabel(r"$\gamma$")
ax.set_yticks([0, 0.5, 1.0])

cbar_left = fig.colorbar(im_left, ax=ax)
cbar_left.set_label(r"$\lambda_{\max}$")
cbar_left.set_ticks([-0.9, -0.6, -0.3, 0, 0.3])

ax.text(-0.2, 1.05, "b)", transform=ax.transAxes, va="top", ha="right")
ax.set_xlim(0.1, 0.7)

# ---------------------------------------------------------------------
# RIGHT PLOT: gamma slice with replicas distance
# ---------------------------------------------------------------------
ax = axes[1]

# Slice df_LLE at g=3
df_slice = df_LLE[df_LLE["g"] == 3].sort_values("gamma")
gamma_vals_right = df_slice["gamma"].values
Z_vals_right = df_slice["Z"].values

gamma_threshold = 0.6

# LLE (black) with vertical line and shaded regions
ax.errorbar(
    gamma_vals_right,
    Z_vals_right,
    color="black",
    fmt="o-",
    lw=2,
    markersize=3,
)

ax.set_xlabel(r"$\gamma$")
ax.set_ylabel(r"$\lambda_{\max}$", color="black")
ax.tick_params(axis="y", colors="black")

ax.axvline(x=gamma_threshold, color="black", linestyle="-.", lw=2)

xmin, xmax = ax.get_xlim()
ax.axvspan(xmin, gamma_threshold, color="firebrick", alpha=0.25, zorder=0)
ax.axvspan(gamma_threshold, xmax, color="darkorange", alpha=0.25, zorder=0)

# Replicas distance (blue) on twin axis
ax2 = ax.twinx()
for N in grouped["N"].unique():
    subset = grouped[grouped["N"] == N]
    ax2.errorbar(
        subset["gamma"],
        subset["avg_d"],
        yerr=subset["std_d"] / 10,
        fmt="o-",
        color="royalblue",
        markersize=3,
        capsize=3,
    )

ax2.set_ylabel("Replicas distance", color="royalblue")
ax2.tick_params(axis="y", colors="royalblue")
ax.set_xlim(0, 1)

ax.set_xticks(
    [0, 0.1, 0.2, 0.4, 0.5, 0.6, 0.8, 0.9, 1]
)
ax.set_xticklabels(
    ["0", "(iv)", "0.2", "0.4", "(iii)", "0.6", "0.8", "(ii)", "1"]
)

ax.text(-0.1, 1.05, "c)", transform=ax.transAxes, va="top", ha="right")

plt.tight_layout()
plt.savefig(
    "PLACEHOLDER/phase_diagram_gamma_bottom.png",
    dpi=1000,
)